In [ ]:
import pandas as pd
import numpy as np

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]
data = data
target = target


In [ ]:
data


In [ ]:
data.shape


In [ ]:
target


In [ ]:
target.shape


In [ ]:
X = data


In [ ]:
Y = target.reshape((506, 1))
Y.shape


In [ ]:
Y[:5]


# Train test split

In [ ]:
%pip install scikit-learn
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=0)


In [ ]:
X_train.shape


In [ ]:
X_test.shape


# Standard scaler

In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
X_train.mean()


In [ ]:
X_train.std()


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
X_train.mean()


In [ ]:
X_train.std()


# Entrainement du model

In [ ]:
def erreur(X, Y, params):

  info = {}

  M = np.dot(X, params['W'])

  P = M + params['B']

  L = np.mean((Y-P) ** 2)

  info['M'] = M
  info['P'] = P

  info['X'] = X
  info['Y'] = Y



  return L, info


In [ ]:
def gradient(info, params):
  grads = {}

  dL_dP = -2 * (info['Y'] - info['P']) # (4, 1)

  dP_dM = 1

  dM_dW = info['X'].T #(3, 4)


  dL_dW = np.dot(dM_dW, dL_dP) * dP_dM   # (4, 1) * (3, 4)  # (3, 4) (4, 1)
  grads['W'] = dL_dW


  dP_dB = 1

  dL_dB = dL_dP * dP_dB #(4, 1) * 1

  dL_dB = np.sum(dL_dB)

  grads['B'] = dL_dB

  return grads


In [ ]:
def train(X, Y, epoch, learning_rate):

  # weights initialization
  np.random.seed(42)
  n_features  = X.shape[1]
  params = {}
  params['W'] = np.random.randn(n_features, 1)
  params['B'] = np.random.randn(1, 1)

  errors = []
  for i in range(epoch):

    # forward
    loss, info = erreur(X, Y, params)
    errors.append(loss)
    print(f'Epoch {i+1} .............. loss : {loss}')

    #backward

    grads = gradient(info, params)


    # update
    for p in params:
      params[p] = params[p] - learning_rate * grads[p]

  return params, errors


In [ ]:
X_train.shape


In [ ]:
np.random.randn(13, 1)


In [ ]:
params, errors = train(X_train, y_train, epoch=500, learning_rate=0.00001)


In [ ]:
import matplotlib.pyplot as plt
plt.plot(list(range(500)), errors)
plt.xlabel('Epochs')
plt.ylabel("Loss")
plt.title("Learning Curve")
plt.show()


# Evaluation sur les données de test

In [ ]:
def predict(X, params):

  M = np.dot(X, params['W'])

  P = M + params['B']

  return P


In [ ]:
def mse(y, pred): # mean squared error
  return np.mean( (y-pred)** 2)

def rmse(y, pred):
  return np.sqrt(np.mean( (y-pred)** 2))

def mae(y, pred): # mean absolute error
  return np.mean(np.abs(y - pred))


In [ ]:
preds = predict(X_test, params)
score_rmse = rmse(y_test, preds)
score_mae = mae(y_test, preds)
print("rmse : ", score_rmse)
print("mae : ", score_mae)


In [ ]:
np.sqrt(2.998)


In [ ]:
# rmse _train = 4.6, rmse_test = 5.78 # overfitting


In [ ]:
params


In [ ]:
# Critère # 5: RM       average number of rooms per dwelling
params["W"][5]


# Comparaison avec sklearn

In [ ]:
from sklearn.linear_model import LinearRegression


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
score_rmse = rmse(y_test, preds)
score_mae = mae(y_test, preds)
print("rmse : ", score_rmse)
print("mae : ", score_mae)


In [ ]:
X_train[5].shape


In [ ]:
params["W"][5]


In [ ]:
params["B"].item()


In [ ]:
plt.scatter(X_train[:, 5], y_train)
plt.plot(X_train[:, 5], params["W"][5] * X_train[:, 5] + params["B"].item(), c="r")
plt.xlabel("Nombre de chambres")
plt.ylabel('Prix')
plt.show()
